In [ ]:
%%writefile session1_seed.py
"""Session 1 seed script: generates one orders dataset and loads it everywhere."""
import argparse
import json
import os
import random
from datetime import datetime, timedelta
import pandas as pd

SEED = 26
CITIES = ["Lahore", "Karachi", "Islamabad", "Rawalpindi", "Faisalabad", "Multan",
          "Peshawar", "Quetta", "Sialkot", "Hyderabad"]
CITY_WEIGHTS = [22, 25, 12, 8, 9, 7, 6, 3, 4, 4]
FIRST = ["Ali", "Ahmed", "Fatima", "Ayesha", "Hassan", "Zainab", "Usman", "Maryam",
         "Bilal", "Hira", "Omar", "Sana", "Hamza", "Amna", "Saad", "Iqra", "Talha",
         "Noor", "Imran", "Mahnoor"]
LAST = ["Khan", "Malik", "Butt", "Sheikh", "Chaudhry", "Qureshi", "Raza", "Siddiqui",
        "Mirza", "Abbasi"]
SEGMENTS = ["Consumer", "Corporate", "Small Business"]
CATEGORIES = {"Electronics": (2500, 90000), "Grocery": (300, 6000), "Fashion": (900, 15000),
              "Home": (1500, 40000), "Beauty": (400, 8000), "Books": (300, 3500)}
STATUSES = ["delivered", "shipped", "pending", "cancelled", "returned"]
STATUS_WEIGHTS = [55, 20, 12, 8, 5]
PAYMENTS = ["card", "cash_on_delivery", "bank_transfer", "wallet"]

START = datetime(2025, 1, 1)
END = datetime(2026, 10, 24, 23, 59)


def make_customers(rng, n=2000):
    rows, movers = [], {}
    for cid in range(1, n + 1):
        signup = START - timedelta(days=rng.randint(0, 900))
        rows.append({
            "customer_id": cid,
            "name": f"{rng.choice(FIRST)} {rng.choice(LAST)}",
            "city": rng.choices(CITIES, CITY_WEIGHTS)[0],
            "segment": rng.choices(SEGMENTS, [70, 15, 15])[0],
            "signup_date": signup.date(),
        })
        if rng.random() < 0.05:
            old = rng.choices([c for c in CITIES if c != rows[-1]["city"]])[0]
            movers[cid] = (old, datetime(2026, 1, 1) + timedelta(days=rng.randint(0, 240)))
    return pd.DataFrame(rows), movers


def make_orders(rng, customers, n=200_000):
    span = int((END - START).total_seconds())
    cust_ids = customers["customer_id"].tolist()
    weights = [min(rng.paretovariate(1.5), 15) for _ in cust_ids]
    picks = rng.choices(cust_ids, weights, k=n)
    rows = []
    for oid, cid in enumerate(picks, start=100001):
        order_ts = START + timedelta(seconds=rng.randint(0, span))
        cat = rng.choice(list(CATEGORIES))
        lo, hi = CATEGORIES[cat]
        qty = rng.choices([1, 2, 3, 4, 5], [60, 22, 10, 5, 3])[0]
        amount = round(rng.uniform(lo, hi) * qty, 2)
        status = rng.choices(STATUSES, STATUS_WEIGHTS)[0]
        days_after = {"pending": 0, "shipped": rng.randint(1, 3), "delivered": rng.randint(2, 8),
                      "cancelled": rng.randint(0, 2), "returned": rng.randint(10, 25)}[status]
        updated = min(order_ts + timedelta(days=days_after, minutes=rng.randint(0, 600)), END)
        rows.append({
            "order_id": oid,
            "customer_id": cid,
            "order_date": order_ts.date(),
            "order_ts": order_ts,
            "category": cat,
            "quantity": qty,
            "amount": amount,
            "status": status,
            "payment_method": rng.choice(PAYMENTS),
            "updated_at": updated,
        })
    return pd.DataFrame(rows).sort_values("order_ts").reset_index(drop=True)


def make_documents(rng, orders, customers, movers, n=5000):
    cust = customers.set_index("customer_id")
    sample = orders.sample(n=n, random_state=SEED).sort_values("order_ts")
    docs = []
    for r in sample.itertuples(index=False):
        doc = {
            "order_id": int(r.order_id),
            "order_date": r.order_ts.isoformat(),
            "status": r.status,
            "amount": float(r.amount),
            "items": [{"category": r.category, "quantity": int(r.quantity)}],
            "payment": {"method": r.payment_method},
            "updated_at": r.updated_at.isoformat(),
        }
        if rng.random() < 0.7:
            c = cust.loc[r.customer_id]
            city = c["city"]
            if r.customer_id in movers and r.order_ts < movers[r.customer_id][1]:
                city = movers[r.customer_id][0]
            doc["customer"] = {"id": int(r.customer_id), "name": c["name"], "city": city}
            if rng.random() < 0.3:
                doc["customer"]["segment"] = c["segment"]
        else:
            doc["customer_id"] = int(r.customer_id)
        if rng.random() < 0.2:
            doc["coupon"] = rng.choice(["WELCOME10", "ELEVEN11", "EID25", "FREESHIP"])
        if rng.random() < 0.1:
            doc["delivery_notes"] = rng.choice(["Call before arriving", "Leave with guard", "Office hours only"])
        if rng.random() < 0.15:
            doc["payment"]["installments"] = rng.choice([3, 6, 12])
        docs.append(doc)
    return docs


def write_files(orders, customers, docs, out="data"):
    os.makedirs(out, exist_ok=True)
    orders.drop(columns=["order_ts"]).to_csv(f"{out}/orders.csv", index=False)
    customers.to_csv(f"{out}/customers.csv", index=False)
    with open(f"{out}/orders.json", "w") as f:
        json.dump(docs, f, indent=1)
    print(f"Wrote {out}/orders.csv ({len(orders):,} rows), {out}/customers.csv, {out}/orders.json ({len(docs):,} docs)")


def load_postgres(url, orders, customers, ro_password):
    from sqlalchemy import create_engine, text
    url = url.replace("postgres://", "postgresql://", 1).replace("postgresql://", "postgresql+psycopg2://", 1)
    engine = create_engine(url)
    with engine.begin() as con:
        con.execute(text("DROP TABLE IF EXISTS orders"))
        con.execute(text("DROP TABLE IF EXISTS customers"))
        con.execute(text("""
            CREATE TABLE customers (
                customer_id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL,
                segment TEXT NOT NULL,
                signup_date DATE NOT NULL
            )"""))
        con.execute(text("""
            CREATE TABLE orders (
                order_id INTEGER PRIMARY KEY,
                customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
                order_date DATE NOT NULL,
                category TEXT NOT NULL,
                quantity INTEGER NOT NULL,
                amount NUMERIC(12, 2) NOT NULL,
                status TEXT NOT NULL,
                payment_method TEXT NOT NULL,
                updated_at TIMESTAMP NOT NULL
            )"""))
    customers.to_sql("customers", engine, if_exists="append", index=False, method="multi", chunksize=1000)
    print("Loading 200k orders into Postgres, about 1 to 3 minutes...")
    orders.drop(columns=["order_ts"]).to_sql("orders", engine, if_exists="append", index=False,
                                             method="multi", chunksize=5000)
    with engine.begin() as con:
        con.execute(text("CREATE INDEX IF NOT EXISTS idx_orders_updated_at ON orders(updated_at)"))
        con.execute(text("CREATE INDEX IF NOT EXISTS idx_orders_order_date ON orders(order_date)"))
        n = con.execute(text("SELECT COUNT(*) FROM orders")).scalar()
        print(f"Postgres: customers={len(customers):,}, orders={n:,}")
        if ro_password:
            db = con.execute(text("SELECT current_database()")).scalar()
            exists = con.execute(text("SELECT 1 FROM pg_roles WHERE rolname = 'student_ro'")).scalar()
            verb = "ALTER" if exists else "CREATE"
            pw = ro_password.replace("'", "''")
            con.execute(text(f"{verb} ROLE student_ro WITH LOGIN PASSWORD '{pw}'"))
            con.execute(text(f'GRANT CONNECT ON DATABASE "{db}" TO student_ro'))
            con.execute(text("GRANT USAGE ON SCHEMA public TO student_ro"))
            con.execute(text("GRANT SELECT ON customers, orders TO student_ro"))
            print("Read-only role student_ro is ready.")


def load_mongo(url, docs):
    from pymongo import MongoClient
    client = MongoClient(url, serverSelectionTimeoutMS=15000)
    col = client["shop"]["orders"]
    col.drop()
    col.insert_many([dict(d) for d in docs])
    col.create_index("status")
    col.create_index("updated_at")
    print(f"MongoDB: shop.orders has {col.count_documents({}):,} documents "
          f"({col.count_documents({'status': 'shipped'}):,} shipped)")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--pg", default=os.getenv("PG_URL"))
    p.add_argument("--mongo", default=os.getenv("MONGO_URL"))
    p.add_argument("--readonly-password", default=os.getenv("RO_PASSWORD"))
    p.add_argument("--orders", type=int, default=200_000)
    p.add_argument("--docs", type=int, default=5000)
    p.add_argument("--out", default="data")
    p.add_argument("--skip-pg", action="store_true")
    p.add_argument("--skip-mongo", action="store_true")
    p.add_argument("--files-only", action="store_true")
    a = p.parse_args()

    rng = random.Random(SEED)
    customers, movers = make_customers(rng)
    orders = make_orders(rng, customers, a.orders)
    docs = make_documents(rng, orders, customers, movers, a.docs)
    write_files(orders, customers, docs, a.out)

    if not (a.files_only or a.skip_pg):
        if not a.pg:
            raise SystemExit("Pass --pg (or set PG_URL), or use --skip-pg")
        load_postgres(a.pg, orders, customers, a.readonly_password)
    if not (a.files_only or a.skip_mongo):
        if not a.mongo:
            raise SystemExit("Pass --mongo (or set MONGO_URL), or use --skip-mongo")
        load_mongo(a.mongo, docs)


if __name__ == "__main__":
    main()

In [ ]:
!pip -q install psycopg2-binary pymongo sqlalchemy pandas
import os
from google.colab import userdata
os.environ["PG_URL"] = userdata.get("PG_URL")
os.environ["MONGO_URL"] = userdata.get("MONGO_URL")
!python session1_seed.py

# 1. **Extraction**

In [1]:
!pip -q install psycopg2-binary pymongo sqlalchemy pandas pyarrow requests
import os, json, glob, requests
import pandas as pd
from datetime import date
from sqlalchemy import create_engine, text
from pymongo import MongoClient
from google.colab import userdata

PG_URL    = userdata.get("PG_URL").replace("postgresql://", "postgresql+psycopg2://", 1)
MONGO_URL = userdata.get("MONGO_URL")
engine    = create_engine(PG_URL)

TODAY = date.today().isoformat()
os.makedirs("raw", exist_ok=True)
print("ok", TODAY)

ok 2026-11-05


In [2]:
customers = pd.read_sql("SELECT * FROM customers", engine)
customers.to_parquet(f"raw/customers_{TODAY}.parquet", index=False)

parts = []
with engine.connect().execution_options(stream_results=True) as con:
    for i, chunk in enumerate(pd.read_sql(text("SELECT * FROM orders ORDER BY order_id"),
                                          con, chunksize=50000), start=1):
        print("chunk", i, len(chunk))
        parts.append(chunk)

orders_pg = pd.concat(parts, ignore_index=True)
orders_pg.to_parquet(f"raw/orders_{TODAY}.parquet", index=False)
print("customers:", len(customers), "| orders:", len(orders_pg))

chunk 1 50000
chunk 2 50000
chunk 3 50000
chunk 4 50000
customers: 2000 | orders: 200000


In [3]:
col  = MongoClient(MONGO_URL)["shop"]["orders"]
docs = list(col.find({}))
with open(f"raw/mongo_orders_{TODAY}.json", "w") as f:
    json.dump(docs, f, default=str)   # default=str turns _id into text
print("mongo docs:", len(docs))

mongo docs: 5000


In [4]:
cities = {"Lahore": (31.55, 74.34), "Karachi": (24.86, 67.01), "Islamabad": (33.68, 73.05)}

for city, (lat, lon) in cities.items():
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={lat}&longitude={lon}&start_date=2025-01-01&end_date=2026-10-24"
           "&daily=temperature_2m_max,precipitation_sum&timezone=Asia/Karachi")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(f"raw/weather_{city}_{TODAY}.json", "w") as f:
        json.dump(r.json(), f)
    print(city, len(r.json()["daily"]["time"]), "days")

Lahore 662 days
Karachi 662 days
Islamabad 662 days


In [5]:
def extract_incremental(last_run):
    q = text("SELECT * FROM orders WHERE updated_at > :t ORDER BY order_id")
    return pd.read_sql(q, engine, params={"t": last_run})

inc = extract_incremental("2026-10-01")
print("rows after 2026-10-01:", len(inc))

rows after 2026-10-01: 8593


In [6]:
for p in sorted(glob.glob("raw/*")):
    print(p, os.path.getsize(p)//1024, "KB")

raw/customers_2026-11-05.parquet 24 KB
raw/mongo_orders_2026-11-05.json 1612 KB
raw/orders_2026-11-05.parquet 5061 KB
raw/weather_Islamabad_2026-11-05.json 16 KB
raw/weather_Karachi_2026-11-05.json 16 KB
raw/weather_Lahore_2026-11-05.json 16 KB


# **2. Transform and Clean**

In [7]:
raw = json.load(open(glob.glob("raw/mongo_orders_*.json")[0]))
m = pd.json_normalize(raw)
print(m.columns.tolist())

# one customer id column
m["customer_id"] = m["customer.id"].fillna(m["customer_id"]).astype(int)

# items is a list with one dict
m["category"] = m["items"].apply(lambda x: x[0]["category"])
m["quantity"] = m["items"].apply(lambda x: x[0]["quantity"])
m["payment_method"] = m["payment.method"]

# same types and date format as Postgres
m["order_date"] = pd.to_datetime(m["order_date"]).dt.normalize()
m["updated_at"] = pd.to_datetime(m["updated_at"])
m["amount"]     = m["amount"].astype(float)
m["customer_city"] = m["customer.city"]       # keep for SCD in Part 3
m["source"] = "app"

cols = ["order_id","customer_id","order_date","category","quantity",
        "amount","status","payment_method","updated_at","source"]
app_df = m[cols + ["customer_city"]].copy()

['_id', 'order_id', 'order_date', 'status', 'amount', 'items', 'updated_at', 'payment.method', 'payment.installments', 'customer.id', 'customer.name', 'customer.city', 'customer_id', 'coupon', 'delivery_notes', 'customer.segment']


In [8]:
web = pd.read_parquet(glob.glob("raw/orders_*.parquet")[0])
web["order_date"] = pd.to_datetime(web["order_date"]).dt.normalize()
web["updated_at"] = pd.to_datetime(web["updated_at"])
web["amount"]     = web["amount"].astype(float)
web["source"]     = "web"
customers = pd.read_parquet(glob.glob("raw/customers_*.parquet")[0])

In [9]:
both = set(web["order_id"]) & set(app_df["order_id"])
print("order_ids in both sources:", len(both))

combined = pd.concat([web[cols], app_df[cols]], ignore_index=True)

# RULE: one row per order_id. Keep latest updated_at.
# Tie: keep the app row (it carries city-at-order-time and the app source flag).
combined["_pri"] = (combined["source"] == "app").astype(int)
orders = (combined.sort_values(["order_id","updated_at","_pri"])
                  .drop_duplicates("order_id", keep="last")
                  .drop(columns="_pri")
                  .reset_index(drop=True))
print(len(combined), "->", len(orders))
print(orders["source"].value_counts())

order_ids in both sources: 5000
205000 -> 200000
source
web    195000
app      5000
Name: count, dtype: int64


In [10]:
valid_status = {"delivered","shipped","pending","cancelled","returned"}
checks = {
  "no null order_id":            orders["order_id"].notna().all(),
  "order_id unique":             orders["order_id"].is_unique,
  "amount > 0":                  (orders["amount"] > 0).all(),
  "customer_id in customers":    orders["customer_id"].isin(customers["customer_id"]).all(),
  "no future order_date":        (orders["order_date"] <= "2026-10-24").all(),
  "status in 5 known values":    orders["status"].isin(valid_status).all(),
}
qc = pd.DataFrame({"check": checks.keys(),
                   "result": ["PASS" if v else "FAIL" for v in checks.values()]})
qc

,check,result
0,no null order_id,PASS
1,order_id unique,PASS
2,amount > 0,PASS
3,customer_id in customers,PASS
4,no future order_date,PASS
5,status in 5 known values,PASS


In [11]:
frames = []
for p in glob.glob("raw/weather_*.json"):
    city = os.path.basename(p).split("_")[1]
    d = json.load(open(p))["daily"]
    frames.append(pd.DataFrame({"city": city,
                                "date": pd.to_datetime(d["time"]),
                                "max_temp": d["temperature_2m_max"],
                                "rain_mm": d["precipitation_sum"]}))
weather = pd.concat(frames, ignore_index=True)
print(weather.shape)

# order city = customer's current city for now
orders = orders.merge(customers[["customer_id","city"]], on="customer_id", how="left")
orders = orders.merge(weather.rename(columns={"date":"order_date"}),
                      on=["city","order_date"], how="left")
print(orders[["max_temp","rain_mm"]].notna().sum())
orders.head()

(1986, 4)
max_temp    117402
rain_mm     117402
dtype: int64


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at,source,city,max_temp,rain_mm
0,100001,261,2026-09-01,Electronics,1,77635.24,delivered,bank_transfer,2026-09-07 14:58:52,web,Faisalabad,NaN,NaN
1,100002,101,2025-02-28,Grocery,1,5562.63,delivered,card,2025-03-05 07:26:15,web,Rawalpindi,NaN,NaN
2,100003,777,2026-07-08,Books,1,2055.15,shipped,card,2026-07-09 23:35:51,web,Rawalpindi,NaN,NaN
3,100004,1509,2025-02-22,Beauty,1,7414.43,delivered,wallet,2025-03-01 21:29:06,web,Lahore,21.9,0.0
4,100005,162,2026-01-12,Books,4,6784.53,delivered,wallet,2026-01-14 02:59:49,web,Peshawar,NaN,NaN


In [12]:
os.makedirs("clean", exist_ok=True)
orders.to_parquet("clean/orders_clean.parquet", index=False)
app_df.to_parquet("clean/app_df.parquet", index=False)
customers.to_parquet("clean/customers.parquet", index=False)
weather.to_parquet("clean/weather.parquet", index=False)

# **3. Design the Warehouse**

In [13]:
d = pd.DataFrame({"date": pd.date_range("2025-01-01", "2026-12-31")})
d["date_key"]    = d["date"].dt.strftime("%Y%m%d").astype(int)
d["day"]         = d["date"].dt.day
d["day_of_week"] = d["date"].dt.day_name()
d["month"]       = d["date"].dt.month
d["month_name"]  = d["date"].dt.month_name()
d["quarter"]     = d["date"].dt.quarter
d["year"]        = d["date"].dt.year
d["is_weekend"]  = d["date"].dt.dayofweek >= 5
dim_date = d[["date_key","date","day","day_of_week","month","month_name","quarter","year","is_weekend"]]
print(len(dim_date))   # must be 730
dim_date.head()

730


,date_key,date,day,day_of_week,month,month_name,quarter,year,is_weekend
0,20250101,2025-01-01,1,Wednesday,1,January,1,2025,False
1,20250102,2025-01-02,2,Thursday,1,January,1,2025,False
2,20250103,2025-01-03,3,Friday,1,January,1,2025,False
3,20250104,2025-01-04,4,Saturday,1,January,1,2025,True
4,20250105,2025-01-05,5,Sunday,1,January,1,2025,True


In [14]:
cats = sorted(orders["category"].unique())
dim_category = pd.DataFrame({"category_key": range(1, len(cats)+1), "category": cats})
dim_category

,category_key,category
0,1,Beauty
1,2,Books
2,3,Electronics
3,4,Fashion
4,5,Grocery
5,6,Home


In [15]:
MOVE_MIN = pd.Timestamp("2026-01-01")
DATA_END = pd.Timestamp("2026-10-24")

app = app_df[["customer_id","order_date","customer_city"]].dropna(subset=["customer_city"]).copy()
app["order_date"] = pd.to_datetime(app["order_date"]).dt.normalize()
app = app.merge(customers[["customer_id","city"]].rename(columns={"city":"current_city"}),
                on="customer_id")

moves = []
for cid, g in app.groupby("customer_id"):
    g = g.sort_values("order_date")
    cur = g["current_city"].iloc[0]
    old = g[g["customer_city"] != cur]
    if old.empty:
        continue
    last_old  = old["order_date"].max()
    new_after = g[(g["customer_city"] == cur) & (g["order_date"] > last_old)]

    lo = max(last_old + pd.Timedelta(days=1), MOVE_MIN)          # earliest possible move
    hi = new_after["order_date"].min() if len(new_after) else DATA_END   # latest possible move
    hi = max(hi, lo)
    move_date = lo + (hi - lo) // 2                              # middle of the window

    moves.append({"customer_id": cid,
                  "old_city": old["customer_city"].iloc[-1],
                  "move_date": move_date,
                  "window_lo": lo, "window_hi": hi})
moves = pd.DataFrame(moves)

print("movers:", len(moves))
print("moves before 2026:", (moves["move_date"] < MOVE_MIN).sum())   # must be 0
moves.head()

movers: 57
moves before 2026: 0


,customer_id,old_city,move_date,window_lo,window_hi
0,5,Quetta,2026-07-20 00:00:00,2026-04-15,2026-10-24
1,103,Hyderabad,2026-05-29 00:00:00,2026-01-01,2026-10-24
2,119,Multan,2026-05-29 00:00:00,2026-01-01,2026-10-24
3,191,Islamabad,2026-05-29 00:00:00,2026-01-01,2026-10-24
4,197,Peshawar,2026-07-07 12:00:00,2026-03-21,2026-10-24


In [24]:
START = pd.Timestamp("2025-01-01")
FAR   = pd.Timestamp("2099-12-31")

mv = moves[["customer_id", "old_city", "move_date"]].copy()
mv["move_date"] = pd.to_datetime(mv["move_date"]).dt.normalize()
# a move on START would leave an empty old row, so push it one day later
mv["move_date"] = mv["move_date"].clip(lower=START + pd.Timedelta(days=1))

base_cols = ["customer_id", "name", "city", "segment", "signup_date"]

# current row for every customer
cur_rows = customers[base_cols].merge(mv[["customer_id", "move_date"]],
                                      on="customer_id", how="left")
cur_rows["valid_from"] = cur_rows["move_date"].fillna(START)
cur_rows["valid_to"]   = FAR
cur_rows["is_current"] = True
cur_rows = cur_rows.drop(columns="move_date")

# old row for movers only
old_rows = customers[base_cols].merge(mv, on="customer_id")
old_rows["city"]       = old_rows["old_city"]
old_rows["valid_from"] = START
old_rows["valid_to"]   = old_rows["move_date"] - pd.Timedelta(days=1)
old_rows["is_current"] = False
old_rows = old_rows.drop(columns=["old_city", "move_date"])

dim_customer = (pd.concat([old_rows, cur_rows], ignore_index=True)
                  .sort_values(["customer_id", "valid_from"]).reset_index(drop=True))
dim_customer.insert(0, "customer_key", range(1, len(dim_customer) + 1))
print("dim_customer rows:", len(dim_customer), "= 2000 +", len(mv))

# ---- checks ----
# 1. every customer starts on START
first = dim_customer.groupby("customer_id")["valid_from"].min()
print("first row not on START:", (first != START).sum())            # must be 0

# 2. no reversed ranges
print("valid_to < valid_from:", (dim_customer["valid_to"] < dim_customer["valid_from"]).sum())  # 0

# 3. every order matches exactly one row
o = orders[["order_id", "customer_id", "order_date"]].copy()
o["order_date"] = pd.to_datetime(o["order_date"]).dt.normalize()
j = o.merge(dim_customer[["customer_id", "valid_from", "valid_to"]], on="customer_id")
j = j[(j["order_date"] >= j["valid_from"]) & (j["order_date"] <= j["valid_to"])]
hits = j.groupby("order_id").size()
print("orders with 0 matches:", len(o) - hits.shape[0])             # must be 0
print("orders with 2+ matches:", (hits > 1).sum())                  # must be 0

dim_customer rows: 2057 = 2000 + 57
first row not on START: 0
valid_to < valid_from: 0
orders with 0 matches: 0
orders with 2+ matches: 0


In [25]:
feb = orders[(orders["order_date"] >= "2026-02-01") & (orders["order_date"] <= "2026-02-28")]
cand = moves[moves["customer_id"].isin(feb["customer_id"])]
cid = cand["customer_id"].iloc[0] if len(cand) else moves["customer_id"].iloc[0]

display(dim_customer[dim_customer.customer_id == cid])
display(feb[feb.customer_id == cid][["order_id","customer_id","order_date","amount"]])

,customer_key,customer_id,name,city,segment,signup_date,valid_from,valid_to,is_current
4,5,5,Hamza Qureshi,Quetta,Consumer,2022-11-10,2025-01-01,2026-07-19,False
5,6,5,Hamza Qureshi,Karachi,Consumer,2022-11-10,2026-07-20,2099-12-31,True


,order_id,customer_id,order_date,amount
7195,107196,5,2026-02-24,6751.23
14032,114033,5,2026-02-26,6758.93
37038,137039,5,2026-02-10,160925.81
71162,171163,5,2026-02-12,1665.86
95592,195593,5,2026-02-09,5370.53
108758,208759,5,2026-02-08,7308.73
154612,254613,5,2026-02-21,370.89
198133,298134,5,2026-02-10,44115.48


In [26]:
base = orders.drop(columns=["city","max_temp","rain_mm"], errors="ignore")

# SCD join: pick the customer row valid on the order date
f = base.merge(dim_customer[["customer_key","customer_id","city","valid_from","valid_to"]],
               on="customer_id")
f = f[(f["order_date"] >= f["valid_from"]) & (f["order_date"] <= f["valid_to"])]
assert len(f) == len(base), f"row count changed: {len(f)} vs {len(base)}"

# weather by city AT ORDER TIME + date
w = weather.rename(columns={"date":"order_date"})
f = f.merge(w[["city","order_date","max_temp","rain_mm"]], on=["city","order_date"], how="left")

# dimension keys
f["date_key"] = f["order_date"].dt.strftime("%Y%m%d").astype(int)
f = f.merge(dim_category, on="category")

fact_orders = f[["order_id","customer_key","customer_id","date_key","category_key","city",
                 "category","quantity","amount","status","payment_method","source",
                 "updated_at","order_date","max_temp","rain_mm"]].reset_index(drop=True)

print(len(fact_orders))                       # 200000
print(fact_orders["order_id"].is_unique)      # True
print(fact_orders[["max_temp","rain_mm"]].notna().sum())
fact_orders.head()

200000
True
max_temp    115457
rain_mm     115457
dtype: int64


,order_id,customer_key,customer_id,date_key,category_key,city,category,quantity,amount,status,payment_method,source,updated_at,order_date,max_temp,rain_mm
0,100001,268,261,20260901,3,Faisalabad,Electronics,1,77635.24,delivered,bank_transfer,web,2026-09-07 14:58:52,2026-09-01,NaN,NaN
1,100002,102,101,20250228,5,Rawalpindi,Grocery,1,5562.63,delivered,card,web,2025-03-05 07:26:15,2025-02-28,NaN,NaN
2,100003,802,777,20260708,2,Rawalpindi,Books,1,2055.15,shipped,card,web,2026-07-09 23:35:51,2026-07-08,NaN,NaN
3,100004,1549,1509,20250222,1,Lahore,Beauty,1,7414.43,delivered,wallet,web,2025-03-01 21:29:06,2025-02-22,21.9,0.0
4,100005,165,162,20260112,2,Peshawar,Books,4,6784.53,delivered,wallet,web,2026-01-14 02:59:49,2026-01-12,NaN,NaN


In [27]:
dim_date.to_parquet("clean/dim_date.parquet", index=False)
dim_category.to_parquet("clean/dim_category.parquet", index=False)
dim_customer.to_parquet("clean/dim_customer.parquet", index=False)
fact_orders.to_parquet("clean/fact_orders.parquet", index=False)

# **4. Export**

In [28]:
import os
os.makedirs("export", exist_ok=True)

def save(df, name):
    df.to_parquet(f"export/{name}.parquet", index=False,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)

cut = pd.Timestamp("2026-10-20")
fact_initial = fact_orders[fact_orders["updated_at"] <= cut]
fact_incr    = fact_orders[fact_orders["updated_at"] >  cut]
print("initial:", len(fact_initial), "| incremental:", len(fact_incr))

save(fact_initial, "fact_orders_initial")
save(fact_incr,    "fact_orders_incr")
save(dim_customer, "dim_customer")
save(dim_date,     "dim_date")
save(dim_category, "dim_category")

!zip -r export.zip export
from google.colab import files
files.download("export.zip")

initial: 197153 | incremental: 2847
  adding: export/ (stored 0%)
  adding: export/fact_orders_initial.parquet (deflated 22%)
  adding: export/dim_customer.parquet (deflated 52%)
  adding: export/dim_date.parquet (deflated 53%)
  adding: export/fact_orders_incr.parquet (deflated 27%)
  adding: export/dim_category.parquet (deflated 46%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>